# Лабораторная работа №7
## Иерархические и сетевые структуры в реляционных БД
**Тема:** Рекурсивные CTE (WITH RECURSIVE), графовые структуры, поиск путей  
**СУБД:** PostgreSQL 17

## Подключение

In [14]:
import psycopg2
import pandas as pd

# Создаём БД если не существует
conn0 = psycopg2.connect(dbname='postgres', user='feedachyou', host='localhost', port=5432)
conn0.autocommit = True
cur0 = conn0.cursor()
cur0.execute("SELECT 1 FROM pg_database WHERE datname = 'logistics_lab'")
if not cur0.fetchone():
    cur0.execute('CREATE DATABASE logistics_lab')
    print('БД создана')
else:
    print('БД уже существует')
conn0.close()

conn = psycopg2.connect(dbname='logistics_lab', user='feedachyou', host='localhost', port=5432)
conn.autocommit = True
cursor = conn.cursor()
print('Connected to logistics_lab')

БД уже существует
Connected to logistics_lab


## Создание таблиц и заполнение данными

In [15]:
cursor.execute("""
DROP TABLE IF EXISTS routes CASCADE;
DROP TABLE IF EXISTS logistics_nodes CASCADE;

CREATE TABLE logistics_nodes (
    node_id      SERIAL PRIMARY KEY,
    name         VARCHAR(100) NOT NULL,
    node_type    VARCHAR(20) CHECK (node_type IN ('warehouse', 'hub', 'store', 'pickup_point')),
    parent_hub_id INT REFERENCES logistics_nodes(node_id)
);

CREATE TABLE routes (
    route_id       SERIAL PRIMARY KEY,
    from_node      INT NOT NULL REFERENCES logistics_nodes(node_id),
    to_node        INT NOT NULL REFERENCES logistics_nodes(node_id),
    distance_km    NUMERIC(8,2) CHECK (distance_km > 0),
    travel_time_min INT CHECK (travel_time_min > 0),
    UNIQUE (from_node, to_node)
);
""")
print('Таблицы созданы')

Таблицы созданы


In [16]:
cursor.execute("""
INSERT INTO logistics_nodes (node_id, name, node_type, parent_hub_id) VALUES
    (1, 'Центральный склад',    'warehouse',    NULL),
    (2, 'Хаб "Север"',          'hub',          1),
    (3, 'Хаб "Юг"',             'hub',          1),
    (4, 'Магазин "Северный"',   'store',        2),
    (5, 'ПВЗ "Южный"',          'pickup_point', 3),
    (6, 'Магазин "Центральный"','store',        1);

INSERT INTO routes (route_id, from_node, to_node, distance_km, travel_time_min) VALUES
    (1, 1, 2, 120, 90),
    (2, 1, 3, 200, 150),
    (3, 2, 4, 30,  20),
    (4, 3, 5, 15,  10),
    (5, 1, 6, 5,   5),
    (6, 2, 6, 80,  55),
    (7, 6, 5, 70,  50);
""")
print('Данные вставлены')

Данные вставлены


## Задание 1 — Все потомки узла (WITH RECURSIVE)

Рекурсивный CTE: начинаем с заданного узла, на каждом шаге присоединяем дочерние узлы через `parent_hub_id`.

In [17]:
df1 = pd.read_sql("""
WITH RECURSIVE descendants AS (
    -- Базовый случай: стартовый узел
    SELECT node_id, name, node_type, parent_hub_id, 0 AS level
    FROM logistics_nodes
    WHERE node_id = 1

    UNION ALL

    -- Рекурсивный шаг: дочерние узлы
    SELECT n.node_id, n.name, n.node_type, n.parent_hub_id, d.level + 1
    FROM logistics_nodes n
    JOIN descendants d ON n.parent_hub_id = d.node_id
)
SELECT node_id, name, node_type, level
FROM descendants
ORDER BY level, name;
""", conn)
df1

/var/folders/zb/dsr0j93j6sgd8wplkp6h6x3m0000gn/T/ipykernel_23938/3902262291.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df1 = pd.read_sql("""


,node_id,name,node_type,level
0,1,Центральный склад,warehouse,0
1,6,"Магазин ""Центральный""",store,1
2,2,"Хаб ""Север""",hub,1
3,3,"Хаб ""Юг""",hub,1
4,4,"Магазин ""Северный""",store,2
5,5,"ПВЗ ""Южный""",pickup_point,2


## Задание 2 — Дерево с отступами и уровнем вложенности

К рекурсивному CTE добавляем визуальный отступ (`repeat('  ', level)`) и сортировку по уровню и имени.

In [18]:
df2 = pd.read_sql("""
WITH RECURSIVE tree AS (
    SELECT node_id, name, node_type, 0 AS level,
           name::TEXT AS sort_path
    FROM logistics_nodes
    WHERE parent_hub_id IS NULL

    UNION ALL

    SELECT n.node_id, n.name, n.node_type, t.level + 1,
           t.sort_path || ' > ' || n.name
    FROM logistics_nodes n
    JOIN tree t ON n.parent_hub_id = t.node_id
)
SELECT
    level,
    repeat('  ', level) || name AS tree_view,
    node_type
FROM tree
ORDER BY sort_path;
""", conn)

for _, row in df2.iterrows():
    print(f"{'  ' * row['level']}{row['tree_view'].strip()} ({row['node_type']})")

Центральный склад (warehouse)
  Магазин "Центральный" (store)
  Хаб "Север" (hub)
  Хаб "Юг" (hub)
    Магазин "Северный" (store)
    ПВЗ "Южный" (pickup_point)


/var/folders/zb/dsr0j93j6sgd8wplkp6h6x3m0000gn/T/ipykernel_23938/979572166.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df2 = pd.read_sql("""


## Задание 3 — Все пути между двумя узлами

Рекурсивный CTE по таблице `routes`. Чтобы избежать циклов — храним массив посещённых узлов и проверяем `NOT (to_node = ANY(visited))`. Глубина ограничена 5 переходами.

In [19]:
df3 = pd.read_sql("""
WITH RECURSIVE paths AS (
    SELECT
        from_node,
        to_node,
        distance_km::NUMERIC          AS distance_km,
        travel_time_min::INT          AS travel_time_min,
        ARRAY[from_node, to_node]     AS visited,
        CAST(from_node AS TEXT) || ' → ' || CAST(to_node AS TEXT) AS path,
        1 AS hops
    FROM routes
    WHERE from_node = 1

    UNION ALL

    SELECT
        r.from_node,
        r.to_node,
        p.distance_km + r.distance_km,
        p.travel_time_min + r.travel_time_min,
        p.visited || r.to_node,
        p.path || ' → ' || CAST(r.to_node AS TEXT),
        p.hops + 1
    FROM routes r
    JOIN paths p ON r.from_node = p.to_node
    WHERE NOT (r.to_node = ANY(p.visited))
      AND p.hops < 5
)
SELECT path, distance_km AS total_km, travel_time_min AS total_min, hops
FROM paths
WHERE to_node = 5
ORDER BY total_km;
""", conn)
df3

/var/folders/zb/dsr0j93j6sgd8wplkp6h6x3m0000gn/T/ipykernel_23938/1035848243.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df3 = pd.read_sql("""


,path,total_km,total_min,hops
0,1 → 6 → 5,75.0,55,2
1,1 → 3 → 5,215.0,160,2
2,1 → 2 → 6 → 5,270.0,195,3


## Задание 4 — Потомки без WITH RECURSIVE (через JOIN)

Три последовательных `LEFT JOIN` по `parent_hub_id` — покрывают глубину до 3 уровней.  
**Ограничения:** фиксированная глубина, нечитаемо при росте дерева, не работает для произвольной вложенности.

In [20]:
df4 = pd.read_sql("""
SELECT
    n1.node_id AS id_1, n1.name AS level_1,
    n2.node_id AS id_2, n2.name AS level_2,
    n3.node_id AS id_3, n3.name AS level_3,
    n4.node_id AS id_4, n4.name AS level_4
FROM logistics_nodes n1
LEFT JOIN logistics_nodes n2 ON n2.parent_hub_id = n1.node_id
LEFT JOIN logistics_nodes n3 ON n3.parent_hub_id = n2.node_id
LEFT JOIN logistics_nodes n4 ON n4.parent_hub_id = n3.node_id
WHERE n1.node_id = 1;
""", conn)
df4

/var/folders/zb/dsr0j93j6sgd8wplkp6h6x3m0000gn/T/ipykernel_23938/2459408908.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df4 = pd.read_sql("""


,id_1,level_1,id_2,level_2,id_3,level_3,id_4,level_4
0,1,Центральный склад,6,"Магазин ""Центральный""",NaN,NaN,None,None
1,1,Центральный склад,3,"Хаб ""Юг""",5.0,"ПВЗ ""Южный""",None,None
2,1,Центральный склад,2,"Хаб ""Север""",4.0,"Магазин ""Северный""",None,None
